# Fine-Tune Qwen2.5 Instruct with QLoRA

Use this notebook on Google Colab.

Default dataset:

```text
fine_tuning_dataset.jsonl
```

Default model:

```text
Qwen/Qwen2.5-3B-Instruct
```

If Colab runs out of VRAM, switch to:

```text
Qwen/Qwen2.5-1.5B-Instruct
```


## Install Dependencies


In [ ]:
!pip install -q -U transformers accelerate datasets peft trl bitsandbytes sentencepiece


## Upload Dataset

Upload `processed/fine_tuning_dataset.jsonl` from your local machine to Colab.


In [ ]:
DATA_PATH = '/content/fine_tuning_dataset.jsonl'


If you prefer Google Drive instead of manual upload, skip the upload cell and use:

```python
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/fine_tuning_dataset.jsonl'
```


In [ ]:
import json
from pathlib import Path

records = []
with open(DATA_PATH, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print("Records:", len(records))
records[0]


In [ ]:
def validate_record(record):
    assert "messages" in record
    messages = record["messages"]
    assert isinstance(messages, list)
    assert len(messages) >= 3
    roles = [message.get("role") for message in messages]
    assert roles[0] == "system"
    assert "user" in roles
    assert roles[-1] == "assistant"
    for message in messages:
        assert message.get("content", "").strip()


for record in records[:100]:
    validate_record(record)

print("Validation passed for first 100 records")


## Load and Validate Dataset


In [ ]:
from datasets import Dataset

raw_dataset = Dataset.from_list(records)
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(train_dataset)
print(eval_dataset)


## Convert to Hugging Face Dataset


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False


## Load Qwen2.5 Model


In [ ]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


train_dataset = train_dataset.map(format_chat, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(format_chat, remove_columns=eval_dataset.column_names)

print(train_dataset[0]["text"][:1200])


## Format Chat Data


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


## Configure LoRA


In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/qwen2_5_shopping_chatbot_lora"
MAX_SEQ_LENGTH = 2048

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,

    fp16=False,
    bf16=True,

    optim="paged_adamw_8bit",
    report_to="none",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    args=training_args,
)

trainer.train()


## Train


In [ ]:
ADAPTER_DIR = "/content/shopping_chatbot_lora_adapter"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved adapter to:", ADAPTER_DIR)


If `SFTTrainer` reports an error for `dataset_text_field` or `max_seq_length`, your `trl` version is newer. Use the alternative cell below.


In [ ]:
!zip -r /content/shopping_chatbot_lora_adapter.zip /content/shopping_chatbot_lora_adapter

from google.colab import files
files.download('/content/shopping_chatbot_lora_adapter.zip')
